# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to explore the [FAIR² dataset](https://doi.org/10.71728/senscience.vq4a-28xa), which contains human protein abundance and annotation data acquired from mass spectrometry of extracellular vesicles isolated from stimulated human mast cells.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We inspect the `record_sets` defined in the schema, and list their `@id`s and fields. Try loading the record set records individually to see their data shapes and field keys.


In [ ]:
# Inspect available record sets. Each is uniquely identified by its '@id'.
record_sets = list(dataset.record_sets)
print("Available record sets and their '@id's:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name'] if 'name' in rs else ''}")
    if 'field' in rs:
        print("  Fields and their '@id's:")
        for f in rs['field']:
            print(f"    - {f['@id']}: {f['name'] if 'name' in f else ''}")
    print()
# For demonstration, pick the first available record set for detailed inspection.
if len(record_sets) > 0:
    example_rs_id = record_sets[0]['@id']
    print(f"\nSample records from record set '{example_rs_id}':")
    for i, rec in enumerate(dataset.records(record_set=example_rs_id)):
        if i >= 3:
            break
        print(rec)
else:
    print("No record sets defined in this Croissant package.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

The following block demonstrates how to extract all record sets into Pandas DataFrames by their `@id`, and inspect the columns.


In [ ]:
# Prepare to extract data from all record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        print(f"Record set '@id': {record_set_id}, columns: {list(df.columns)}")
        dataframes[record_set_id] = df
        print(df.head())
    else:
        print(f"Record set '@id': {record_set_id} contains no records.")

# For analysis, select the main protein quantitation table, if present (by common biological naming heuristics)
protein_table_id = None
for rs in record_sets:
    if 'protein' in rs.get('name', '').lower() or 'abundance' in rs.get('name', '').lower() or 'quant' in rs.get('name', '').lower():
        protein_table_id = rs['@id']
        break
if not protein_table_id and len(record_set_ids) > 0:
    protein_table_id = record_set_ids[0]
print(f"\nSelected main data table record set '@id': {protein_table_id}")
main_df = dataframes.get(protein_table_id, pd.DataFrame())
print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Let's process the main protein quantitation DataFrame. We'll filter records with high molecular weights (MW), normalize the MW, and group by a descriptive column (e.g., gene or protein family if available).

Replace field and group names with actual values found in your main table's columns. We always reference these by their `@id` when available.


In [ ]:
# Identify possible numeric and group-able fields by column names or Croissant field '@id'
main_columns = list(main_df.columns)
possible_numeric_fields = [c for c in main_columns if 'weight' in c.lower() or 'mw' in c.lower() or 'abundance' in c.lower() or 'count' in c.lower() or 'coverage' in c.lower()]
print(f"Possible numeric fields: {possible_numeric_fields}")

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # E.g., 'MW', '@id_MW', etc.
else:
    numeric_field = main_columns[0] if len(main_columns) > 0 else None

# Find a group-able field (often protein description, gene, or class)
possible_group_fields = [c for c in main_columns if 'desc' in c.lower() or 'gene' in c.lower() or 'family' in c.lower()]
group_field = possible_group_fields[0] if possible_group_fields else None

print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

# Remove missing values and convert to numeric
df_eda = main_df.copy()
if numeric_field:
    df_eda = df_eda[pd.to_numeric(df_eda[numeric_field], errors='coerce').notnull()]  # Remove non-numeric
    df_eda[numeric_field] = pd.to_numeric(df_eda[numeric_field], errors='coerce')
    threshold = df_eda[numeric_field].mean()  # Use the mean as threshold for demonstration
    filtered_df = df_eda[df_eda[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # If group_field exists, group and display summary
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}': mean of {numeric_field}")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between numeric fields.

Example: Histogram of MW distribution, and a boxplot if grouping field exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

# Boxplot of numeric field by group field
if numeric_field and group_field and group_field in main_df.columns:
    plt.figure(figsize=(10, 5))
    # Limit to top 10 groups to keep plot readable
    top_groups = main_df[group_field].value_counts().index[:10]
    sns.boxplot(x=group_field, y=numeric_field, data=main_df[main_df[group_field].isin(top_groups)])
    plt.title(f"{numeric_field} by {group_field} (Top 10 groups)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² mass spectrometry dataset using the `mlcroissant` library. We:
- Loaded metadata and records using the Croissant schema
- Inspected record sets and fields by their `@id`s
- Loaded the main protein quantitation table into a DataFrame
- Filtered and normalized protein fields such as MW or abundance
- Visualized the distribution and grouping of selected variables

This approach can be extended by selecting other record sets, fields or performing deeper domain analysis as required for proteomics and biomarker discovery.